# 🚀 PageLoop — Week 3B: Convert the Databricks Method

Use PageLoop to learn the method, then convert it to the assigned project.

The converted notebook must stop at:

```text
one project Bronze demo table
→ one project lineage demo view
```

The full project Bronze layer belongs to Week 4.

## 🎯 Week-3 outcome

By the end of this notebook, every intern should be able to:

- find uploaded files in a Databricks Volume;
- understand how Spark SQL reads CSV and JSON files;
- create temporary views;
- inspect table structure and column types;
- display table contents;
- explain grain and business keys;
- compare physical rows with distinct business records;
- inspect values and distributions;
- identify simple data-quality concerns;
- check relationships between files;
- create one small managed Delta preview;
- explain what belongs to Week 3 and what belongs to Week 4.

> **The objective is understanding—not speed.**

## 🧩 Databricks cell languages

Databricks allows different cell languages inside one notebook.

Use the cell-language dropdown to choose:

- **Python** for PySpark;
- **SQL** for Spark SQL;
- **File system** for `%fs` commands.

You can also place a magic command at the top of a cell:

```text
%python
%sql
%fs
```

In this notebook:

- PySpark is used for loading files and basic DataFrame inspection;
- Spark SQL is used for most exploration and analysis;
- selected activities are shown in both styles so interns can compare them.

## 🧭 Notebook map

| Section | What you will do |
|---|---|
| 1 | Check the uploaded files |
| 2 | Create Spark SQL views |
| 3 | Inspect schemas |
| 4 | Display table contents |
| 5 | Understand grain |
| 6 | Count records |
| 7 | Inspect values |
| 8 | Find simple data concerns |
| 9 | Check relationships |
| 10 | Ask one business question |
| 11 | Create a Bronze preview |
| 12 | Capture evidence and explain the work |

# 1. Prepare Databricks

Before running this notebook:

1. Open your Databricks workspace.
2. Attach **Serverless notebook compute**.
3. Create a Volume named `pageloop`.
4. Upload these three Week-3 files:

```text
loans.csv.gz
books.csv
branches.json
```

Recommended location:

```text
/Volumes/workspace/default/pageloop/
```

Week-10 event files are not required today.

## 💡 Tip — Know where things live

| Item | Correct place |
|---|---|
| Notebook | Databricks Workspace |
| Full data files | Unity Catalog Volume |
| Screenshots | GitHub repository |
| Weekly log | GitHub repository |
| Full working dataset | Do not commit to GitHub |

A notebook contains instructions.

A Volume contains data.

# 2. Check the uploaded files

Before loading data, confirm that the files are visible.

The following command lists the contents of the PageLoop Volume.

In [0]:
%fs
ls /Volumes/workspace/default/bingemetrics

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/bingemetrics/content_catalog.json,content_catalog.json,1123734,1784956676000
dbfs:/Volumes/workspace/default/bingemetrics/sessions.parquet,sessions.parquet,7057470,1784956678000
dbfs:/Volumes/workspace/default/bingemetrics/subscriptions.csv,subscriptions.csv,2511788,1784956676000
dbfs:/Volumes/workspace/default/bingemetrics/users.csv,users.csv,1832448,1784956676000


### Expected files

```text
loans.csv.gz
books.csv
branches.json
```

If one is missing, stop and upload it before continuing.

> **Professional habit:** Always confirm the source files before writing queries.

# 3. Create Spark SQL views

A Spark SQL view gives a file a simple table-like name.

Instead of repeatedly referring to a long file path, we can write:

```sql
SELECT * FROM loans
```

We will create one temporary view for each source file.

## 3.1 Create the `loans` view

The loan file is a compressed CSV.

`header = true` tells Spark that the first row contains column names.

`inferSchema = true` asks Spark to detect common data types.

### What this Python block does

This cell:

1. reads the compressed CSV file;
2. creates a PySpark DataFrame named `loans`;
3. creates a temporary SQL view named `loans`.

Use the cell-language dropdown and select **Python** before running it.

In [0]:
# Load the loan file as a PySpark DataFrame
subscriptions = spark.read.csv(
    "/Volumes/workspace/default/bingemetrics/subscriptions.csv",
    header=True,
    inferSchema=True
)

# Make the DataFrame available to Spark SQL
subscriptions.createOrReplaceTempView("subscriptions")

## 3.2 Create the `books` view

### What this Python block does

This cell reads the books CSV and creates both:

- a PySpark DataFrame named `books`;
- a Spark SQL temporary view named `books`.

In [0]:
# Load the books file
users = spark.read.csv(
    "/Volumes/workspace/default/bingemetrics/users.csv",
    header=True,
    inferSchema=True
)

# Make it available to Spark SQL
users.createOrReplaceTempView("users")

## 3.3 Create the `branches` view

### What this Python block does

This cell reads the JSON branch file and creates both:

- a PySpark DataFrame named `branches`;
- a Spark SQL temporary view named `branches`.

In [0]:
# Load the branches file
Content_catalog = spark.read.json(
    "/Volumes/workspace/default/bingemetrics/content_catalog.json"
)

# Make it available to Spark SQL
Content_catalog.createOrReplaceTempView("Content_catalog")

## 3.4 Confirm that the views exist

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bingemetrics_bronze_demo_subscriptions,false
default,bingemetrics_week03_lineage_demo_view,false
,content_catalog,true
,subscriptions,true
,users,true


You should see temporary views named:

```text
loans
books
branches
```

These views exist for the current notebook session.

# 3A. Confirm the created DataFrames

At this point, three PySpark DataFrames should exist:

```text
loans
books
branches
```

Use the following short Python cell to display their names and column counts.

In [0]:
# Confirm the three DataFrames and their column counts
print("subscriptions columns:", len(subscriptions.columns))
print("users columns:", len(users.columns))
print("Content_catalog columns:", len(Content_catalog.columns))

subscriptions columns: 9
users columns: 8
Content_catalog columns: 14


This is a simple existence check.

It does not replace schema inspection, row counts or table previews.

# 4. Inspect the schema

A schema describes the structure of a dataset.

It tells us:

- column names;
- data types;
- possible identifiers;
- date and timestamp fields;
- numeric measures;
- nullable fields.

We will inspect each view separately.

## 4.1 Loans schema

### PySpark method — print the DataFrame schema

This is the quickest way to inspect DataFrame columns and data types.

In [0]:
# Show loan column names and data types
subscriptions.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_code: string (nullable = true)
 |-- billing_cycle: string (nullable = true)
 |-- period_start_date: date (nullable = true)
 |-- period_end_date: date (nullable = true)
 |-- lifecycle_status: string (nullable = true)
 |-- auto_renew_flag: boolean (nullable = true)
 |-- cancellation_reason_group: string (nullable = true)



### Spark SQL method — describe the SQL view

The SQL version displays the same structure in table form.

In [0]:
%sql
DESCRIBE subscriptions;

col_name,data_type,comment
subscription_id,string,null
user_id,string,null
plan_code,string,null
billing_cycle,string,null
period_start_date,date,null
period_end_date,date,null
lifecycle_status,string,null
auto_renew_flag,boolean,null
cancellation_reason_group,string,null


### What to notice

Look for:

- `loan_id` — likely business key;
- `member_code` — member identifier;
- `book_id` and `branch_id` — relationship fields;
- `checkout_ts`, `due_ts`, `return_ts` — time fields;
- `status` — category field;
- `loan_period_days`, `fine_amount`, `net_fine` — numeric fields.

## 4.2 Users schema

In [0]:
%sql
DESCRIBE users;

col_name,data_type,comment
user_id,string,null
signup_date,date,null
signup_cohort,string,null
age_band,string,null
region_band,string,null
preferred_device_segment,string,null
acquisition_channel,string,null
user_status,string,null


## 4.3 Content catalog schema

In [0]:
%sql
DESCRIBE content_catalog;

col_name,data_type,comment
available_from_date,string,null
available_to_date,string,null
catalog_status,string,null
content_id,string,null
content_type,string,null
creator_label,string,null
duration_seconds,bigint,null
genre,string,null
is_platform_original,boolean,null
language,string,null


## 💡 Tip — Compare with Week 2

Open the Week-2 data dictionary.

Compare it manually with the actual schema.

Ask:

- Are the expected columns present?
- Did Spark detect the expected data types?
- Is any column missing?
- Is any extra column present?
- Does the proposed business key actually exist?

Do not automate this comparison yet.

First learn to read the schema yourself.

# 5. Display the table contents

A schema tells us the structure.

The actual rows tell us how the data looks.

Always inspect a few rows before writing analytical queries.

## 5.1 Display loan records

### PySpark method — display DataFrame rows

This uses the `subscriptions` DataFrame created earlier.

In [0]:
# Display the first 10 subscriptions records
display(subscriptions.limit(10))

subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null


### Spark SQL method — display the same rows

The SQL query below reads from the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM subscriptions
LIMIT 10;

subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null


### Look carefully

Notice:

- how IDs are formatted;
- how timestamps appear;
- whether `return_ts` can be empty;
- the values used in `status`;
- whether numeric fields contain decimals or negatives.

## 5.2 Display users records

### PySpark method — display the users DataFrame

This confirms that the CSV was loaded into the `users` DataFrame.

In [0]:
# Display the first 10 users records
display(users.limit(10))

user_id,signup_date,signup_cohort,age_band,region_band,preferred_device_segment,acquisition_channel,user_status
U000001,2025-03-01,2025-Q1,25-34,South-Urban,TV-first,Partner,ACTIVE
U000002,2024-05-22,2024-Q2,18-24,East-Urban,TV-first,Campaign,ACTIVE
U000003,2025-05-01,2025-Q2,35-44,East-Urban,Mobile-first,Organic,ACTIVE
U000004,2024-08-16,2024-Q3,18-24,North-Urban,Mobile-first,Organic,ACTIVE
U000005,2024-10-26,2024-Q4,55+,North-Urban,TV-first,Campaign,ACTIVE
U000006,2025-04-02,2025-Q2,25-34,West-Urban,Mobile-first,Organic,ACTIVE
U000007,2025-05-23,2025-Q2,25-34,West-Urban,Multi-device,Organic,ACTIVE
U000008,2025-03-28,2025-Q1,25-34,West-Urban,TV-first,Organic,ACTIVE
U000009,2024-02-21,2024-Q1,25-34,North-Urban,Multi-device,Partner,DORMANT
U000010,2025-04-14,2025-Q2,25-34,North-Urban,Desktop-first,Partner,ACTIVE


### Spark SQL method — display the users view

The SQL query below reads the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM users
LIMIT 10;

user_id,signup_date,signup_cohort,age_band,region_band,preferred_device_segment,acquisition_channel,user_status
U000001,2025-03-01,2025-Q1,25-34,South-Urban,TV-first,Partner,ACTIVE
U000002,2024-05-22,2024-Q2,18-24,East-Urban,TV-first,Campaign,ACTIVE
U000003,2025-05-01,2025-Q2,35-44,East-Urban,Mobile-first,Organic,ACTIVE
U000004,2024-08-16,2024-Q3,18-24,North-Urban,Mobile-first,Organic,ACTIVE
U000005,2024-10-26,2024-Q4,55+,North-Urban,TV-first,Campaign,ACTIVE
U000006,2025-04-02,2025-Q2,25-34,West-Urban,Mobile-first,Organic,ACTIVE
U000007,2025-05-23,2025-Q2,25-34,West-Urban,Multi-device,Organic,ACTIVE
U000008,2025-03-28,2025-Q1,25-34,West-Urban,TV-first,Organic,ACTIVE
U000009,2024-02-21,2024-Q1,25-34,North-Urban,Multi-device,Partner,DORMANT
U000010,2025-04-14,2025-Q2,25-34,North-Urban,Desktop-first,Partner,ACTIVE


## 5.3 Display content catalog records

### PySpark method — display the Content_catalog DataFrame

This confirms that the JSON file was loaded into the `Content_catalog` DataFrame.

In [0]:
# Display the first 10 branch records
display(Content_catalog.orderBy("content_id").limit(10))

available_from_date,available_to_date,catalog_status,content_id,content_type,creator_label,duration_seconds,genre,is_platform_original,language,maturity_band,release_year,series_collection_id,title_label
2017-01-01,null,LIMITED,C000001,VIDEO,CreatorGroup_0001,2526,Sports Stories,false,Telugu,A,2017,null,Parallel Notes 0001
2002-01-01,null,AVAILABLE,C000002,MUSIC,CreatorGroup_0002,210,Jazz,false,Telugu,U/A 16+,2002,null,Coastal Rhythms 0002
2013-01-01,null,AVAILABLE,C000003,MUSIC,CreatorGroup_0003,260,Devotional,false,Hindi,U,2013,null,Open Voices 0003
2016-01-01,null,AVAILABLE,C000004,MUSIC,CreatorGroup_0004,166,Indie Pop,true,Kannada,U,2016,null,Midnight Journeys 0004
2002-01-01,null,AVAILABLE,C000005,PODCAST,CreatorGroup_0005,4517,Careers,false,Spanish,U/A 16+,2002,null,Neon Archives 0005
1999-01-01,null,LIMITED,C000006,VIDEO,CreatorGroup_0006,6503,Documentary,false,Hindi,U/A 7+,1999,COL00001,Silver Horizons 0006
2026-01-01,null,AVAILABLE,C000007,VIDEO,CreatorGroup_0007,3063,Comedy,false,Hindi,U/A 13+,2026,null,Urban Stories 0007
1993-01-01,null,AVAILABLE,C000008,VIDEO,CreatorGroup_0008,2346,Sports Stories,true,English,U/A 13+,1993,null,Hidden Patterns 0008
2026-01-01,null,AVAILABLE,C000009,VIDEO,CreatorGroup_0009,5528,Drama,false,Telugu,U,2026,COL00002,Solar Frames 0009
2012-01-01,null,AVAILABLE,C000010,VIDEO,CreatorGroup_0010,2415,Comedy,false,Spanish,A,2012,null,Quiet Circuits 0010


### Spark SQL method — display the branches view

The SQL query below reads the temporary view created from the same DataFrame.

In [0]:
%sql
SELECT *
FROM Content_catalog
ORDER BY content_id
LIMIT 10;

available_from_date,available_to_date,catalog_status,content_id,content_type,creator_label,duration_seconds,genre,is_platform_original,language,maturity_band,release_year,series_collection_id,title_label
2017-01-01,null,LIMITED,C000001,VIDEO,CreatorGroup_0001,2526,Sports Stories,false,Telugu,A,2017,null,Parallel Notes 0001
2002-01-01,null,AVAILABLE,C000002,MUSIC,CreatorGroup_0002,210,Jazz,false,Telugu,U/A 16+,2002,null,Coastal Rhythms 0002
2013-01-01,null,AVAILABLE,C000003,MUSIC,CreatorGroup_0003,260,Devotional,false,Hindi,U,2013,null,Open Voices 0003
2016-01-01,null,AVAILABLE,C000004,MUSIC,CreatorGroup_0004,166,Indie Pop,true,Kannada,U,2016,null,Midnight Journeys 0004
2002-01-01,null,AVAILABLE,C000005,PODCAST,CreatorGroup_0005,4517,Careers,false,Spanish,U/A 16+,2002,null,Neon Archives 0005
1999-01-01,null,LIMITED,C000006,VIDEO,CreatorGroup_0006,6503,Documentary,false,Hindi,U/A 7+,1999,COL00001,Silver Horizons 0006
2026-01-01,null,AVAILABLE,C000007,VIDEO,CreatorGroup_0007,3063,Comedy,false,Hindi,U/A 13+,2026,null,Urban Stories 0007
1993-01-01,null,AVAILABLE,C000008,VIDEO,CreatorGroup_0008,2346,Sports Stories,true,English,U/A 13+,1993,null,Hidden Patterns 0008
2026-01-01,null,AVAILABLE,C000009,VIDEO,CreatorGroup_0009,5528,Drama,false,Telugu,U,2026,COL00002,Solar Frames 0009
2012-01-01,null,AVAILABLE,C000010,VIDEO,CreatorGroup_0010,2415,Comedy,false,Spanish,A,2012,null,Quiet Circuits 0010


## 🧠 Intern checkpoint 1

Complete these statements:

```text
The main transaction file is session.parquet.
One row appears to represent one playback session.
The likely business key is session_id.
The main date field is session_start_ts.
The main status field is end_reason.
```

Do not move ahead until the answers make sense.

# 6. Understand the grain

## What is grain?

**Grain means what one row represents.**

Examples:

- one loan record;
- one book;
- one branch;
- one payment;
- one delivery;
- one incident.

For PageLoop:

| View | Expected grain |
|---|---|
| `loans` | one physical loan source record |
| `books` | one book record |
| `branches` | one branch record |

The grain must be understood before counting business activity.

# 7. Count the physical records

Start with a simple row count for each view.

## PySpark method — count one DataFrame

This counts the physical rows in the main loan DataFrame.

In [0]:
# Count physical loan rows
subscriptions_row_count = subscriptions.count()
print("Physical loan rows:", subscriptions_row_count)

Physical loan rows: 35000


## Spark SQL method — count all three views

The SQL query below produces a compact source summary.

In [0]:
%sql
SELECT 'subscriptions' AS source, COUNT(*) AS records
FROM subscriptions
UNION ALL
SELECT 'users', COUNT(*)
FROM users
UNION ALL
SELECT 'Content_catalog', COUNT(*)
FROM Content_catalog;

source,records
subscriptions,35000
users,25000
Content_catalog,3000


This query tells us how many physical rows Spark loaded from each source.

For the loan file, the expected physical row count is:

```text
90,279
```

# 8. Compare rows with distinct business keys

A physical row count is not always the same as a business record count.

The proposed loan business key is `loan_id`.

Let us count distinct loan IDs.

## PySpark method — count distinct business keys

This uses simple DataFrame operations to count unique `subscription_id` values.

In [0]:
# Count distinct loan IDs
distinct_subscriptions_count = (
    subscriptions.select("subscription_id")
         .distinct()
         .count()
)

print("Distinct loan IDs:", distinct_subscriptions_count)

Distinct loan IDs: 34980


## Spark SQL method — compare both values together

The SQL query below is more compact for analytical comparison.

In [0]:
%sql
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT subscription_id) AS distinct_subscriptions
FROM subscriptions;

physical_rows,distinct_subscriptions
35000,34980


### Expected PageLoop result

```text
Physical rows:      90,279
Distinct loan IDs:  90,000
Difference:            279
```

This means the file contains repeated business keys.

> **Professional interpretation:** There are 90,279 source records representing 90,000 distinct loans.

# 9. Display repeated business keys

Now identify a few repeated `loan_id` values.

In [0]:
%sql
SELECT
  subscription_id,
  COUNT(*) AS occurrences
FROM subscriptions
GROUP BY subscription_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

subscription_id,occurrences
S0031118,2
S0032623,2
S0031581,2
S0032445,2
S0033121,2
S0027731,2
S0025151,2
S0032481,2
S0030915,2
S0030841,2


## 🧠 Intern checkpoint 2

Explain this in one sentence:

```text
The row count is higher than the distinct-loan count because
of duplicate rows.
```

This is the first grain-related discovery.

# 10. Inspect important values

Before looking for errors, understand the normal values in the file.

## 10.1 Status distribution

### PySpark method — group and count

This groups the DataFrame by `status` and counts records.

In [0]:
# Count records by lifecycle status
display(
    subscriptions.groupBy("lifecycle_status")
         .count()
         .orderBy("count", ascending=False)
)

lifecycle_status,count
ACTIVE,25040
EXPIRED,9900
PAUSED_FOREVER,60


### Spark SQL method — perform the same analysis

The SQL version below gives the same business result.

In [0]:
%sql
SELECT
  lifecycle_status,
  COUNT(*) AS records
FROM subscriptions
GROUP BY lifecycle_status
ORDER BY records DESC;

lifecycle_status,records
ACTIVE,25040
EXPIRED,9900
PAUSED_FOREVER,60


### Why this matters

A distribution helps us understand:

- common categories;
- rare categories;
- possible spelling differences;
- unexpected values;
- whether one category dominates the data.

## 10.2 Checkout date range

In [0]:
%sql
SELECT
  MIN(period_start_date) AS first_subscription,
  MAX(period_start_date) AS latest_subscription
FROM subscriptions;

first_subscription,latest_subscription
2023-05-16,2026-02-01


## 10.3 Loan-period range

In [0]:
%sql
SELECT
  MIN(DATEDIFF(period_end_date, period_start_date)) AS minimum_subscription_days,
  MAX(DATEDIFF(period_end_date, period_start_date)) AS maximum_subscription_days,
  AVG(DATEDIFF(period_end_date, period_start_date)) AS average_subscription_days
FROM subscriptions;

minimum_subscription_days,maximum_subscription_days,average_subscription_days
-2078,911,421.7251428571429


## 10.4 Fine-value range

In [0]:
%sql
SELECT
  MIN(DATEDIFF(period_end_date, period_start_date)) AS minimum_subscription_days,
  MAX(DATEDIFF(period_end_date, period_start_date)) AS maximum_subscription_days,
  AVG(DATEDIFF(period_end_date, period_start_date)) AS average_subscription_days
FROM subscriptions
WHERE period_end_date >= period_start_date;

minimum_subscription_days,maximum_subscription_days,average_subscription_days
90,911,426.5524627720504


# 11. Find simple data concerns

Week 3 is about observation.

We are not cleaning the data yet.

We are only asking:

> **What should the Week-4 and Week-5 pipeline handle?**

## 11.1 Missing member codes

In [0]:
%sql
SELECT COUNT(*) AS missing_user_ids
FROM subscriptions
WHERE user_id IS NULL
   OR TRIM(user_id) = '';

missing_user_ids
0


Expected result:

```text
1,611 records
```

A missing member code may affect member-level analysis.

## 11.2 Negative loan periods

In [0]:
%sql
SELECT COUNT(*) AS negative_subscription_periods
FROM subscriptions
WHERE DATEDIFF(period_end_date, period_start_date) < 0;

negative_subscription_periods
80


Expected result:

```text
180 records
```

A negative loan period is logically suspicious.

## 11.3 Return before checkout

In [0]:
%sql
SELECT COUNT(*) AS end_before_start
FROM subscriptions
WHERE period_end_date IS NOT NULL
  AND period_end_date < period_start_date;

end_before_start
80


Expected result:

```text
252 records
```

This is an example of an impossible timestamp sequence.

## 11.4 Display a few suspicious records

In [0]:
%sql
SELECT
  subscription_id,
  user_id,
  period_start_date,
  period_end_date,
  DATEDIFF(period_end_date, period_start_date) AS subscription_period_days,
  lifecycle_status
FROM subscriptions
WHERE DATEDIFF(period_end_date, period_start_date) < 0
   OR period_end_date < period_start_date;

subscription_id,user_id,period_start_date,period_end_date,subscription_period_days,lifecycle_status
S0025023,U000023,2023-09-03,2020-01-01,-1341,EXPIRED
S0025349,U000349,2025-07-07,2020-01-01,-2014,EXPIRED
S0025577,U000577,2025-04-28,2020-01-01,-1944,EXPIRED
S0025579,U000579,2025-03-20,2020-01-01,-1905,EXPIRED
S0025740,U000740,2024-12-25,2020-01-01,-1820,EXPIRED
S0025789,U000789,2024-07-29,2020-01-01,-1671,EXPIRED
S0025938,U000938,2025-03-09,2020-01-01,-1894,EXPIRED
S0025992,U000992,2024-06-26,2020-01-01,-1638,EXPIRED
S0026107,U001107,2025-03-25,2020-01-01,-1910,EXPIRED
S0026288,U001288,2024-08-16,2020-01-01,-1689,EXPIRED


## 🧠 Intern checkpoint 3

Choose one issue and explain:

```text
I found some subscription records where the period_end_date is earlier than the period_start_date. It could affect subscription duration calculations and business reports. It should be handled during the Data Quality work before loading data into the Silver layer
```

Suggested answer structure:

> “I found ________. It could affect ________. It should be handled during Silver or Data Quality work.”

# 12. Check relationships between files

The loan file contains:

- `book_id`;
- `branch_id`.

A good relationship means the referenced value exists in the related file.

## 12.1 Check the book relationship

In [0]:
%sql
SELECT COUNT(*) AS invalid_user_references
FROM subscriptions s
LEFT JOIN users u
    ON s.user_id = u.user_id
WHERE u.user_id IS NULL;

invalid_user_references
65


Expected result:

```text
558 invalid book references
```

These loan rows point to a book that cannot be found in the books view.

## 12.2 Display a few invalid book references

In [0]:
%sql
SELECT
  s.subscription_id,
  s.user_id,
  s.plan_code,
  s.lifecycle_status
FROM subscriptions s
LEFT JOIN users u
  ON s.user_id = u.user_id
WHERE u.user_id IS NULL
LIMIT 20;

subscription_id,user_id,plan_code,lifecycle_status
S0024608,U024608,Standard,ACTIVE
S0024633,U024633,Standard,ACTIVE
S0024638,U024638,Basic,ACTIVE
S0024677,U024677,Standard,ACTIVE
S0024689,U024689,Premium,ACTIVE
S0024704,U024704,Standard,ACTIVE
S0024710,U024710,Student,ACTIVE
S0024736,U024736,Standard,ACTIVE
S0024757,U024757,AudioPlus,ACTIVE
S0024765,U024765,Basic,ACTIVE


## 12.3 Check the branch relationship

In [0]:
%sql
SELECT COUNT(*) AS invalid_user_references
FROM subscriptions s
LEFT JOIN users u
  ON s.user_id = u.user_id
WHERE u.user_id IS NULL;

invalid_user_references
65


Expected result:

```text
0 invalid branch references
```

Zero is still valuable evidence because the relationship was tested.

# 13. Understand the join effect

An inner join keeps only matching records.

If 558 loan rows do not find a book, those rows will not survive an inner join.

In [0]:
%sql
SELECT COUNT(*) AS matched_subscription_rows
FROM subscriptions s
INNER JOIN users u
  ON s.user_id = u.user_id;

matched_subscription_rows
34960


### Expected connection

```text
90,279 physical loan rows
−   558 invalid book references
= 89,721 matched loan rows
```

This demonstrates why relationship checks matter before building dashboards.

# 14. Ask one business question

The library manager asks:

> Which subscription plan has the highest number of subscriptions?

We will join loans with branches and count the records.

In [0]:
%sql
SELECT
    plan_code,
    COUNT(*) AS subscription_records,
    COUNT(DISTINCT subscription_id) AS distinct_subscriptions
FROM subscriptions
GROUP BY plan_code
ORDER BY subscription_records DESC;

plan_code,subscription_records,distinct_subscriptions
Standard,9765,9764
Basic,8359,8359
Premium,7677,7677
Student,4912,4912
AudioPlus,4287,4285


### How to read this result

Write:

```text
Highest-activity zone:
Number of physical records:
Number of distinct loans:
One-line observation:
```

Then add this limitation:

> Known data issues have not yet been corrected, so this is an exploratory result—not a trusted Gold KPI.

# 15. Preview the Bronze idea

The Week-3 flow is:

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demonstration table
→ one lineage demonstration view
```

Only one Bronze demonstration table is created here. The complete multi-table Bronze layer belongs to Week 4.

## 15.1 Create one Bronze demo table

This managed Delta table uses the main `loans` entity, preserves the source columns and adds only basic ingestion metadata.

No deduplication, correction or Silver transformation is performed.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bingemetrics_bronze_demo_subscriptions
USING DELTA
AS
SELECT
    *,
    current_timestamp() AS ingested_at,
    '/Volumes/workspace/default/bingemetrics/subscriptions.csv' AS source_file
FROM subscriptions;

num_affected_rows,num_inserted_rows


> This is a Week-3 learning table—not the official PageLoop Bronze layer.

# 16. Confirm and display the demo table

In [0]:
%sql
SHOW TABLES IN workspace.default LIKE 'bingemetrics_bronze_demo_subscriptions';

database,tableName,isTemporary
default,bingemetrics_bronze_demo_subscriptions,false


In [0]:
%sql
SELECT *
FROM workspace.default.bingemetrics_bronze_demo_subscriptions
LIMIT 10;

subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group,ingested_at,source_file
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z,/Volumes/workspace/default/bingemetrics/subscriptions.csv


Look for:

```text
ingested_at
source_file
```

# 17. Perform one source-to-demo count check

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM subscriptions) AS source_rows,
  (SELECT COUNT(*) 
   FROM workspace.default.bingemetrics_bronze_demo_subscriptions) AS demo_rows;

source_rows,demo_rows
35000,35000


Expected:

```text
source_rows = demo_rows
```

This is a simple Week-3 confidence check. Full reconciliation belongs to Week 4.

# 18. Inspect Delta table details

In [0]:
%sql
DESCRIBE DETAIL workspace.default.bingemetrics_bronze_demo_subscriptions;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1570968b-1200-4f8f-a62d-caf9eec31dde,workspace.default.bingemetrics_bronze_demo_subscriptions,null,,2026-07-30T13:52:35.234Z,2026-07-30T13:52:37.000Z,List(),List(),1,155473,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Notice fields such as `format`, `location`, `createdAt`, `lastModified` and `numFiles`.

# 19. Inspect Delta table history

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.bingemetrics_week03_lineage_demo_view
AS
SELECT
  subscription_id,
  user_id,
  plan_code,
  billing_cycle,
  period_start_date,
  period_end_date,
  lifecycle_status,
  auto_renew_flag,
  cancellation_reason_group,
  ingested_at
FROM workspace.default.bingemetrics_bronze_demo_subscriptions;

| Concept | Question answered |
|---|---|
| Schema | What columns and data types exist? |
| Relationship | How do business entities connect? |
| History | What operations changed this Delta table? |
| Lineage | Which governed objects feed or use another object? |

# 20. Create a lineage demonstration view

In [0]:
%sql
SELECT *
FROM workspace.default.bingemetrics_week03_lineage_demo_view
LIMIT 10;

subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group,ingested_at
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z


In [0]:
%sql
SELECT *
FROM workspace.default.bingemetrics_week03_lineage_demo_view
LIMIT 20;

subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group,ingested_at
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null,2026-07-30T13:52:35.528Z
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null,2026-07-30T13:52:35.528Z


In [0]:
%sql
DESCRIBE HISTORY workspace.default.bingemetrics_bronze_demo_subscriptions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-07-30T13:52:37.000Z,73138203965680,rsanjana3028@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1058286027562886),4961828b-d46b-4e22-b2a9-9e725d762ba2,0730-124550-weijvznl-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 155473, numDeletionVectorsRemoved -> 0, numOutputRows -> 35000, numOutputBytes -> 155473)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-30T13:29:36.000Z,73138203965680,rsanjana3028@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""comment"":""The table contains data related to subscription metrics, including details about user subscriptions, billing cycles, and lifecycle statuses. It can be used to analyze subscription trends, monitor renewal behaviors, and understand cancellation reasons. The data includes identifiers for users and subscriptions, as well as timestamps for when the data was ingested.""})",null,null,01f18c1a-abab-15dd-85fd-581e472e565f,null,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-30T13:05:10.000Z,73138203965680,rsanjana3028@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1058286027562886),009dc874-5368-485f-8e10-fcc5a2b2eb53,0730-124550-weijvznl-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 155473, numDeletionVectorsRemoved -> 0, numOutputRows -> 35000, numOutputBytes -> 155473)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-30T13:01:48.000Z,73138203965680,rsanjana3028@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1058286027562886),6bf2d5a4-4e34-49c4-a986-ff288624d580,0730-124550-weijvznl-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 35000, numOutputBytes -> 155473)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


The governed lineage path is:

```text
pageloop_week03_bronze_demo_loans
                 ↓
pageloop_week03_lineage_demo_view
```

# 21. View lineage in Catalog Explorer

1. Click **Catalog**.
2. Open `workspace`.
3. Open `default`.
4. Select `pageloop_week03_lineage_demo_view`.
5. Open **Lineage**.
6. Choose **See lineage graph** when available.
7. Identify `pageloop_week03_bronze_demo_loans` as the upstream object.
8. Capture one screenshot.

## 🧠 Intern checkpoint

Explain:

```text
Files → DataFrames → temporary views → exploration
→ one Bronze demo table → one lineage demo view
```

The raw project files were uploaded to a Databricks Volume and loaded into Spark as DataFrames. Temporary SQL views were created to explore the data, inspect the schema, analyze counts and distributions, identify simple data-quality issues, and answer a business question. A single Bronze Delta table (bingemetrics_bronze_demo_subscriptions) was then created to demonstrate raw data ingestion. Finally, a lineage demo view (bingemetrics_week03_lineage_demo_view) was created from the Bronze table to illustrate how data flows through the pipeline.

# 22. Week-3 boundary

## Completed

- source-file inspection;
- PySpark DataFrame creation and display;
- temporary SQL views;
- schema, grain, counts and values;
- simple data concerns;
- relationship checks;
- one business question;
- one Bronze demo table;
- one count check;
- Delta detail and history;
- one lineage demo view;
- Catalog Explorer lineage walkthrough.

## Deferred to Week 4

- official Bronze tables for every source;
- repeatable ingestion;
- complete reconciliation;
- schema handling policy;
- rerun behaviour;
- full Bronze evidence;
- production naming and controls.

# 23. Evidence checklist

```text
screenshots/week03_01_source_files.png
screenshots/week03_02_dataframes.png
screenshots/week03_03_schemas.png
screenshots/week03_04_grain_counts_values.png
screenshots/week03_05_relationship_checks.png
screenshots/week03_06_bronze_demo.png
screenshots/week03_07_delta_history.png
screenshots/week03_08_lineage_graph.png
```

# 24. Final intern defence

Every intern should explain:

1. Files, DataFrames and temporary views.
2. Schema, grain and business keys.
3. Physical rows versus distinct keys.
4. Values, ranges and data concerns.
5. Business relationships and joins.
6. What a managed Delta table is.
7. Why only one demo table was created.
8. What `DESCRIBE DETAIL` shows.
9. What `DESCRIBE HISTORY` shows.
10. What lineage means.
11. What is deferred to Week 4.

# 🎉 Week-3 Databricks foundation complete

```text
Volume files
→ PySpark DataFrames
→ temporary Spark SQL views
→ exploration
→ one Bronze demo table
→ one downstream lineage view
```

This is the correct Week-3 stopping point.

> Week 3 teaches how the data behaves. Week 4 builds the repeatable Bronze foundation.

# Part 2 — Project Conversion Studio

You have now seen the complete Week-3 method.

The next step is to convert:

```text
PageLoop files
PageLoop columns
PageLoop grain
PageLoop keys
PageLoop relationships
PageLoop question
```

into the real structure of your assigned project.

# 20. Prepare the conversion inputs

Provide the AI assistant with:

```text
1. This 03B notebook
2. Your project brief PDF
3. Your actual Data Pack ZIP
4. Your Week-2 data dictionary
```

The project title alone is not enough.

The AI must inspect the real files and fields.

# 21. Ask for a mapping before code

The AI must first produce a mapping table:

| PageLoop concept | Assigned-project equivalent |
|---|---|
| `loans.csv.gz` | main transaction file |
| `books.csv` | first reference file |
| `branches.json` | second reference file |
| `loan_id` | business key |
| `checkout_ts` | main date field |
| `status` | category/status field |
| book relationship | first genuine relationship |
| branch relationship | second genuine relationship |
| PageLoop Volume | project Volume |
| PageLoop preview table | project preview table |

Review this mapping before accepting any generated notebook.

# 22. Master conversion prompt

Copy the following prompt with all project files.

```text
Convert the attached PageLoop Week-3 notebook into a professional,
intern-facing Spark SQL notebook for my assigned data-engineering project.

AUTHORITATIVE INPUTS

1. The attached 03B notebook defines the Week-3 learning sequence,
   explanation quality, code style, evidence and Week-3 boundary.

2. The project brief defines the business context, intended entities,
   relationships and business questions.

3. The Data Pack ZIP is the physical authority for real filenames,
   folders, formats and columns.

4. The Week-2 data dictionary defines the documented grain,
   business keys and assumptions.

FIRST INSPECT — DO NOT GENERATE CODE IMMEDIATELY

Return a mapping table containing:

• actual source filename;
• file format;
• source purpose;
• one-row grain;
• business key;
• important date field;
• category or status field;
• numeric fields worth inspecting;
• reference datasets;
• genuine relationships;
• primary Bronze-preview entity.

DO NOT INVENT

• files;
• paths;
• formats;
• columns;
• data types;
• grain;
• keys;
• relationships;
• record counts;
• expected values;
• business findings.

If a required item is missing or contradictory, stop and return:

CONVERSION BLOCKED

Explain:

• the exact issue;
• the affected notebook section;
• the missing or conflicting input;
• the exact remediation required.

NOTEBOOK STYLE

The notebook is for third-year engineering interns using Databricks
for the first time.

Use Spark SQL as the main implementation language.

Also preserve a small number of short PySpark cells for:

• loading source files;
• creating DataFrames;
• creating temporary SQL views;
• displaying DataFrame rows;
• printing schemas;
• showing one or two equivalent DataFrame operations.

For selected activities, show both:

• PySpark method;
• Spark SQL method.

Clearly label what each code block is trying to do.

Use:

• %fs only to list uploaded files;
• short Python cells for file loading and DataFrame inspection;
• %sql to create temporary views;
• %sql to inspect schemas;
• %sql to display records;
• %sql for counts, distributions, checks, joins and the Bronze preview;
• one clear idea per code cell;
• short professional headings;
• context before every query;
• interpretation after every important result;
• tips and intern checkpoints;
• direct, readable code.

Do not use:

• Python helper functions or advanced Python abstractions;
• try/except;
• type annotations;
• configuration frameworks;
• path-resolution utilities;
• nested dictionaries;
• reusable ingestion frameworks;
• automated report DataFrames;
• automatic PASS/FAIL systems;
• long PySpark chains;
• advanced engineering abstractions.

PRESERVE THESE LEARNING SECTIONS

1. notebook mission and context;
2. Databricks preparation;
3. file inventory;
4. creation of one Spark SQL view per source;
5. confirmation of created views;
6. schema inspection for every source;
7. table-content display for every source;
8. grain explanation;
9. physical record counts;
10. distinct business-key count;
11. repeated-key display;
12. category or status distribution;
13. date range;
14. numeric-value range;
15. missing-value check;
16. one logical or numeric concern;
17. one timestamp or sequence concern when applicable;
18. display of suspicious records;
19. genuine relationship checks;
20. display of invalid references;
21. join-consequence explanation;
22. one simple business question;
23. one managed Delta Bronze preview;
24. source-to-preview count check;
25. preview display;
26. Week-3 versus Week-4 boundary;
27. evidence checklist;
28. final intern defence.

CONVERSION RULES

Replace all PageLoop-specific:

• names;
• file paths;
• filenames;
• formats;
• views;
• columns;
• keys;
• measurements;
• relationships;
• expected values;
• business conclusions;
• preview-table name.

Use this preview naming pattern:

workspace.default.<project_short_name>_bronze_preview_<primary_entity>

Exclude Week-10 streaming files from Week-3 work.

When a PageLoop check does not apply:

• do not invent an equivalent;
• mark it Not applicable;
• explain why;
• choose another simple, genuine check only when supported by the data.

RETURN

1. A complete project-specific IPYNB.
2. A short Databricks setup note.
3. A conversion summary.
4. A list of items interns must verify manually.
5. Any blocked or uncertain items.

The final notebook must be clean, engaging, professional,
Spark-SQL-first and implementation-ready.
```

# 23. Review the generated notebook

Before importing it into Databricks, check:

- Are the real filenames used?
- Does every column exist?
- Does the key actually identify the business entity?
- Are relationships genuine?
- Are there any invented expected values?
- Is the code mainly Spark SQL?
- Are explanations clear before and after queries?
- Are PageLoop references fully removed?

# 24. Run and verify in Databricks

Interns must:

1. import the converted notebook;
2. attach Serverless notebook compute;
3. upload the real project files;
4. update the direct Volume path;
5. run one cell at a time;
6. verify every result;
7. correct AI mistakes;
8. capture genuine screenshots;
9. complete the Week-3 log;
10. commit the verified notebook.

# 25. Remove PageLoop leftovers

Search the converted notebook for:

```text
PageLoop
pageloop
loans
loan_id
books
branches
checkout_ts
member_code
book_id
branch_id
90,279
90,000
558
279
```

Every PageLoop-specific reference must be removed unless the assigned project genuinely uses the same name.

# 26. AI Transparency Note

Complete:

```text
AI tool used:
Purpose:
Files provided:
What AI converted:
Files manually verified:
Columns manually verified:
Keys manually verified:
Relationships manually verified:
AI errors found:
Corrections made:
How the notebook was tested:
What every intern can explain without AI:
```

# 27. Project-conversion acceptance gate

The converted notebook is ready only when:

- [ ] all files are real;
- [ ] all columns exist;
- [ ] Spark SQL is the main language;
- [ ] code blocks remain short and readable;
- [ ] headings and explanations are professional;
- [ ] grain and keys are correct;
- [ ] values come from actual execution;
- [ ] relationships are genuine;
- [ ] suspicious records can be displayed;
- [ ] the Bronze preview reconciles;
- [ ] no PageLoop result remains;
- [ ] evidence is captured;
- [ ] every intern can explain the notebook.

# 🎉 Conversion studio complete

PageLoop provides the method.

Your Data Pack provides the truth.

AI provides a first draft.

Your team provides the engineering judgment.

```text
UNDERSTAND → CONVERT → RUN → VERIFY → CORRECT → EXPLAIN
```

> **The notebook becomes your work only after you verify it.**

# 28. Final conversion authority

The generated project notebook must include all of the following.

## Source and DataFrame foundation

- actual Volume path;
- actual source-file listing;
- one PySpark DataFrame per required Week-3 file;
- clear comments explaining each load;
- one display cell per DataFrame;
- temporary Spark SQL view creation.

## Exploration foundation

- schema inspection;
- table-content display;
- grain;
- physical row counts;
- distinct business-key counts;
- repeated-key check;
- category or status distribution;
- date range;
- numeric range;
- simple data concerns;
- genuine relationship checks;
- one business question.

## Bronze foundation

- one managed Delta Bronze table per core source;
- source columns preserved;
- `ingested_at`;
- `source_file`;
- no Silver cleaning;
- source-to-Bronze reconciliation;
- Bronze sample display;
- `DESCRIBE DETAIL`;
- `DESCRIBE HISTORY`.

## Lineage foundation

- one simple downstream exploration view built from Bronze tables;
- a visual explanation of the flow;
- Catalog Explorer lineage instructions;
- lineage screenshot requirement;
- explanation of lineage versus relationships and Delta history.

# 29. Required project-specific naming

Use simple, consistent names.

Example:

```text
workspace.default.fitpulse_bronze_activity
workspace.default.fitpulse_bronze_member
workspace.default.fitpulse_bronze_device
workspace.default.fitpulse_week03_activity_view
```

Replace `fitpulse` with the actual project short name.

Do not retain PageLoop names.

# 30. Final conversion acceptance gate

The project-specific notebook is accepted only when:

- [ ] real files are listed from the real Volume;
- [ ] every required source becomes a DataFrame;
- [ ] every DataFrame is displayed;
- [ ] every DataFrame has a temporary SQL view;
- [ ] schemas are inspected;
- [ ] SQL table content is displayed;
- [ ] grain and keys are explained;
- [ ] counts come from actual execution;
- [ ] values and ranges are explored;
- [ ] checks use real columns;
- [ ] relationships are genuine;
- [ ] one Bronze table exists per core source;
- [ ] Bronze tables include ingestion metadata;
- [ ] source and Bronze counts reconcile;
- [ ] Delta detail and history are inspected;
- [ ] one downstream exploration view demonstrates lineage;
- [ ] the Catalog Explorer lineage graph is captured;
- [ ] PageLoop references are removed;
- [ ] AI-generated assumptions are manually verified;
- [ ] every intern can defend the notebook.

# 🎉 Week-3B conversion complete

The common notebook provides the method.

The project Data Pack provides the truth.

The project team provides the judgment.

```text
INSPECT
→ MAP
→ CONVERT
→ RUN
→ VERIFY
→ CORRECT
→ EXPLAIN
```

The final output is not called “converted” merely because AI produced it.

It becomes a valid project notebook only after the team executes and verifies every section in Databricks.

# 28. Final Week-3 conversion authority

The project notebook must include real files, DataFrames, displays, temporary views, schema, grain, counts, values, concerns, relationships, one business question, exactly one Bronze demo table, one count check, Delta detail/history, one lineage demo view and one lineage screenshot.

It must not create the complete Bronze layer.

# 29. Project-specific demo naming

Example:

```text
workspace.default.fitpulse_week03_bronze_demo_activity
workspace.default.fitpulse_week03_lineage_demo_view
```

Use the main event or transaction entity. Replace `fitpulse` with the actual project short name.

# 30. Final acceptance gate

- [ ] real files used
- [ ] DataFrames created and displayed
- [ ] SQL views created and displayed
- [ ] schema, grain, keys and counts explained
- [ ] values and concerns inspected
- [ ] relationships validated
- [ ] one business question answered
- [ ] exactly one Bronze demo table created
- [ ] source and demo counts match
- [ ] Delta detail and history shown
- [ ] one lineage demo view created
- [ ] lineage screenshot captured
- [ ] no full Bronze layer created
- [ ] PageLoop names removed
- [ ] every intern can defend the work

# 🎉 Week-3B conversion complete

```text
MAP → CONVERT → RUN → VERIFY → CORRECT → EXPLAIN
```

The converted notebook is valid only after successful Databricks execution and team verification.